# Peer-to-Peer Dataset Federation Demo
This notebook shows how a researcher can query, from a plain Jupyter notebook, Parquet files spread across several institutions as if they were all local, with no centralized infrastructure.

Two Parquet files are used for this demo, already present in `node/data/`:
- `yellow_tripdata_2026-01.parquet`
- `yellow_tripdata_2026-02.parquet`

They come from the public NYC taxi trip data: https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page

For the full architecture (Rust node, iroh, gossip, HTTP API), see [system_architecture.md](../../doc/system_architecture.md).

## Prerequisites

Before running this notebook, at leat one Rust node must be running locally (it exposes the HTTP API on `localhost:8080`):

```bash
cd peer-dataset-federation/node
INSTITUTION=<institution> cargo run -- peer
```

See [installation.md](../../doc/installation.md) for details on launching (multiple peers, `BOOTSTRAP_PEERS`, etc.).

## 1. Connecting to the local node

`P2PClient` is the HTTP wrapper to the Rust node. `P2PDataset` is the layer used in this notebook: local cache, file discover, loading into DataFrames.

In [ ]:
import logging
import pandas as pd
from p2p.client import P2PClient  # HTTP wrapper: talks to the Rust node
from p2p.dataset import P2PDataset  # Cache + query layer: files(), load(), query(), federate()

# Show INFO-level logs (cache hits/misses, fetch calls) directly in the notebook
logging.basicConfig(level=logging.INFO)

# Connect to the local node
client = P2PClient("http://localhost:8080")

# Main entry point: everything below goes through this object
dataset = P2PDataset(client)

## 2. Exploring the files available on the network

`files()` queries the local node, which answers from the manifests received via gossip (its own + every other peer's). `files_df()` shows the same information as a readable table.

In [ ]:
dataset.files()

In [ ]:
dataset.files_df()

In [ ]:
# max_columns_shown limits how many columns are shown per file
dataset.files_df(2)

## 3. Loading a single file: `load()`

`load()` fetches the file (from the local cache if already downloaded, otherwise over the network) and loads it directly into a `pandas.DataFrame`.